In [ ]:
# =========================================
# 1. Imports
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [ ]:
# =========================================
# 2. Load dataset
# =========================================
# Preferred: processed file from EDA
PROCESSED_PATH = Path("../data/processed/airlines_delay_eda.csv")
RAW_PATH = Path("../data/raw/airlines_delay.csv")

if PROCESSED_PATH.exists():
    DATA_PATH = PROCESSED_PATH
elif RAW_PATH.exists():
    DATA_PATH = RAW_PATH
else:
    raise FileNotFoundError(
        "No dataset found. Expected one of these paths:\n"
        f"- {PROCESSED_PATH}\n"
        f"- {RAW_PATH}"
    )

df = pd.read_csv(DATA_PATH)

print("Loaded from:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())


In [ ]:
# =========================================
# 3. Prepare dataset
# =========================================
# If raw dataset is used, create DepartureHour as fallback.
# Note: Time is minutes since midnight (0=00:00, 1439=23:59).
if "DepartureHour" not in df.columns and "Time" in df.columns:
    df["DepartureHour"] = df["Time"] // 60

# Drop columns not used in this model
drop_columns = ["id", "Flight", "Time", "DepartureMinute", "IsWeekend", "LengthBin", "Route"]
existing_drop_columns = [col for col in drop_columns if col in df.columns]

df = df.drop(columns=existing_drop_columns, errors="ignore")

print("Columns after preparation:")
print(df.columns.tolist())
display(df.head())


In [ ]:
# =========================================
# 4. Define features and target
# =========================================
TARGET = "Delay"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
display(y.value_counts())
display((y.value_counts(normalize=True) * 100).round(2))


In [ ]:
# =========================================
# 5. Define feature types
# =========================================
# IsWeekend is excluded: it is redundant with DayOfWeek (already encoded below).
categorical_features = [col for col in ["Airline", "AirportFrom", "AirportTo", "DayOfWeek"] if col in X.columns]
numeric_features = [col for col in ["Length", "DepartureHour"] if col in X.columns]

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)


In [ ]:
# =========================================
# 6. Train-test split
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


In [ ]:
# =========================================
# 7. Preprocessing pipeline
# =========================================
# Note: OHE on AirportFrom + AirportTo produces ~586 sparse columns (293 airports each).
# Plus ~18 Airline columns and 7 DayOfWeek columns.
# Total feature space is large but manageable for LogisticRegression.
# Tree-based models in notebook 03 will handle high cardinality more naturally.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor


In [ ]:
# =========================================
# 8. Baseline model pipeline
# =========================================
# class_weight=None (default) because target is only mildly imbalanced (55/45).
# We will revisit with class_weight='balanced' in notebook 03 if needed.
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

baseline_model


In [ ]:
# =========================================
# 9. Train baseline model
# =========================================
baseline_model.fit(X_train, y_train)

print("Baseline model trained successfully.")


In [ ]:
# =========================================
# 10. Predictions
# =========================================
y_pred = baseline_model.predict(X_test)
y_proba = baseline_model.predict_proba(X_test)[:, 1]

predictions_df = X_test.copy()
predictions_df["actual"] = y_test.values
predictions_df["predicted"] = y_pred
predictions_df["probability_delay"] = y_proba

display(predictions_df.head())


In [ ]:
# =========================================
# 11. Evaluate baseline model
# =========================================
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_proba)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

display(metrics_df)
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# =========================================
# 12. Confusion matrix
# =========================================
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

display(cm_df)

plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix - Baseline Model")
plt.colorbar()
plt.xticks([0, 1], ["Pred 0", "Pred 1"])
plt.yticks([0, 1], ["Actual 0", "Actual 1"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 13. ROC curve
# =========================================
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("ROC Curve - Baseline Model")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 14. Inspect predicted probabilities
# =========================================
predictions_sorted = predictions_df.sort_values("probability_delay", ascending=False)
display(predictions_sorted.head(20))


In [ ]:
# =========================================
# 15. Save baseline model
# =========================================
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "baseline_model.pkl"
joblib.dump(baseline_model, MODEL_PATH)

print(f"Baseline model saved to: {MODEL_PATH}")


In [ ]:
# =========================================
# 16. Save baseline metrics
# =========================================
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = OUTPUT_DIR / "baseline_metrics.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "baseline_predictions_sample.csv"

metrics_df.to_csv(METRICS_PATH, index=False)
predictions_df.head(100).to_csv(PREDICTIONS_PATH, index=False)

print(f"Saved metrics to: {METRICS_PATH}")
print(f"Saved prediction sample to: {PREDICTIONS_PATH}")


In [ ]:
# =========================================
# 17. Final summary for presentation
# =========================================
print("BASELINE MODEL SUMMARY")
print("----------------------")
print("Model: Logistic Regression")
print(f"Dataset used: {DATA_PATH}")
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print()
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print()
print("Features used:")
for col in X.columns:
    print("-", col)
